# Wfo Mag7 Sma Optimization

Interactive notebook version of this workflow. Edit the parameters and rerun cells to experiment.


In [1]:
from __future__ import annotations

from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
if not (repo_root / 'quant_orchestrator').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from dataclasses import dataclass
from itertools import product
from time import perf_counter
from typing import Any

import pandas as pd

from quant_orchestrator.experiments import WindowSpec, build_walk_forward_windows
from quant_orchestrator.platforms.backtesting_frameworks.backtesting_py.sma_crossover import (
    run_sma_crossover_backtest as run_backtesting_py_runner,
)
from quant_orchestrator.platforms.backtesting_frameworks.nautilus.sma_crossover import (
    run_sma_crossover_backtest as run_nautilus_runner,
)
from quant_orchestrator.platforms.backtesting_frameworks.shared import (
    MAG7_SYMBOLS,
    build_sma_crossover_frame,
    load_price_frame,
    normalize_session_label,
)
from quant_orchestrator.platforms.backtesting_frameworks.zipline.sma_crossover import (
    run_sma_crossover_backtest as run_zipline_runner,
)
from quant_orchestrator.strategy import summarize_backtest


FAST_WINDOWS = (10, 20, 50)
SLOW_WINDOWS = (100, 150, 200)


@dataclass(frozen=True)
class FrameworkRun:
    raw_result: object
    summary: pd.DataFrame
    equity: pd.Series


def build_sma_frame(prices: pd.DataFrame, *, fast_window: int, slow_window: int) -> pd.DataFrame:
    frame = build_sma_crossover_frame(prices, fast_window=fast_window, slow_window=slow_window)
    frame.attrs['fast_window'] = fast_window
    frame.attrs['slow_window'] = slow_window
    return frame


def _wrap_runner(runner):
    def _run(frame: pd.DataFrame, *, symbol: str, capital_base: float):
        fast_window = int(frame.attrs.get('fast_window', 50))
        slow_window = int(frame.attrs.get('slow_window', 200))
        return runner(
            frame.loc[:, ['open', 'high', 'low', 'close', 'volume']].copy(),
            symbol=symbol,
            fast_window=fast_window,
            slow_window=slow_window,
            capital_base=capital_base,
        )

    return _run


run_backtesting_py = _wrap_runner(run_backtesting_py_runner)
run_zipline = _wrap_runner(run_zipline_runner)
run_nautilus = _wrap_runner(run_nautilus_runner)


Walk-forward schedule:
WalkForwardWindow(train_start=Timestamp('2020-01-01 00:00:00'), train_end=Timestamp('2025-12-31 00:00:00'), test_start=Timestamp('2026-01-01 00:00:00'), test_end=Timestamp('2026-12-31 00:00:00'))

Native WFO support:
     framework native_optimize
backtesting.py             yes
       zipline              no
      nautilus              no



/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/backtesting/_plotting.py:50: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support (e.g. PyCharm, Spyder IDE). Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(


/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbol", "country_code"], group_keys=False).apply(
/home/jlee153232/miniconda3/envs/quant-orchestrator/lib/python3.12/site-packages/zipline/assets/asset_writer.py:321: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mappings.groupby(["symbo

...


2026 test portfolio summary:


,framework,symbol,fast_window,slow_window,bars,trades,final_equity,total_return,max_drawdown,elapsed_seconds
0,backtesting.py,MAG7,10,100,118,16,97577.46,-0.0242,-0.0263,0.0759
1,zipline,MAG7,20,100,118,32,97938.60,-0.0206,-0.2248,0.2766
2,nautilus,MAG7,20,100,118,26,98230.91,-0.0177,-0.0241,0.0719



Native backtesting.py portfolio summary:


,framework,symbol,bars,trades,final_equity,total_return,max_drawdown,elapsed_seconds
0,backtesting.py-native,MAG7,118,1,100000.0,0.0,0.0,2.7929



Native engine snapshots:
[backtesting.py]
=== AAPL ===
Start                2026-01-02 00:00:00
End                  2026-06-23 00:00:00
Equity Final [$]            14261.924262
Return [%]                      -0.16653
Sharpe Ratio                         0.0
Max. Drawdown [%]              -3.312401
# Trades                               3
dtype: object

=== AMZN ===
Start                2026-01-02 00:00:00
End                  2026-06-23 00:00:00
Equity Final [$]            13674.614203
Return [%]                     -4.277701
Sharpe Ratio                         0.0
Max. Drawdown [%]              -5.789218
# Trades                               2
dtype: object

=== GOOGL ===
Start                2026-01-02 00:00:00
End                  2026-06-23 00:00:00
Equity Final [$]             14201.67446
Return [%]                     -0.588279
Sharpe Ratio                         0.0
Max. Drawdown [%]              -4.577265
# Trades                               2
dtype: object

=== META ==